# Research Orientation Notebook

This notebook bridges the college-level build sequence into the research layer. The goal is not to reproduce a frontier paper end to end. The goal is to learn a disciplined loop: state the claim, identify the bottleneck, name the mechanism, define a baseline, choose a metric, and run the smallest toy replication that could fail.

## Motivation

Research notebooks are for falsifiable understanding, not paper-summary theater. For each paper, extract these six pieces before touching code:

- **Claim:** what the paper says gets better.
- **Bottleneck:** what limitation or cost the paper targets.
- **Mathematical mechanism:** what equation, factorization, constraint, or estimator changes.
- **Objective or architecture change:** what the implementation actually swaps in or adds.
- **Evidence:** what empirical result is supposed to support the claim.
- **Limitations:** where the evidence is narrow, confounded, or expensive.

### Filled paper lab example

Using *Attention Is All You Need* [@attention_is_all_you_need] as the first example:

| Field | Filled note |
|---|---|
| Claim | Self-attention plus positional information can replace recurrence for sequence transduction while improving parallelism. |
| Bottleneck | Recurrent sequence models have long path lengths and weak hardware parallelism. |
| Mathematical mechanism | Scaled dot-product attention forms weighted combinations of token states; multi-head attention repeats that mechanism in several learned subspaces. |
| Objective or architecture change | Replace recurrent encoder-decoder stacks with self-attention blocks and feed-forward layers while keeping a familiar sequence-to-sequence training setup. |
| Evidence | Translation benchmarks and training-cost comparisons. |
| Limitations | The paper does not remove the quadratic attention score matrix, and the full system bundles several changes at once. |
| Toy replication | Compare an order-blind baseline against an order-aware attention variant on a task where only position-sensitive retrieval can work. |

## Hypothesis

If the variant adds an order-sensitive aggregation mechanism, it should outperform an order-blind baseline on a first-token retrieval task. This is the level of claim a research orientation notebook should make: small, testable, and easy to invalidate.

## Baseline

The baseline embeds each token, averages across positions, and predicts the first token from that pooled vector. Because averaging is permutation-invariant, the baseline has no direct way to preserve the identity of position 0.

The variant keeps the same token embeddings and classifier head, but replaces uniform pooling with learned attention pooling plus positional embeddings. That is the paper-lab pattern: change one mechanism, keep the comparison tight, and say exactly what changed.

## Metric

Use one primary metric and one paired diagnostic on the same evaluation examples:

- **Primary metric:** validation accuracy.
- **Paired diagnostic:** per-example negative log-likelihood gain of the variant over the baseline.

A paired comparison matters because both models are judged on the same examples. That reduces noise and makes the evidence easier to interpret.

## Mathematical derivation

Let a sequence be `x = (x_1, ..., x_T)` with label `y = x_1`. Let `e(x_t)` be a token embedding.

The mean-pooling baseline computes

$$h_{\text{base}}(x) = \frac{1}{T} \sum_{t=1}^{T} e(x_t), \qquad p_{\text{base}}(y \mid x) = \operatorname{softmax}(W h_{\text{base}} + b).$$

This representation is permutation-invariant: any reordering of the tokens leaves `h_base` unchanged. So on a task where the label depends on a specific position, the baseline discards the needed information.

The variant adds positional embeddings `p_t` and attention weights

$$s_t = q^\top (e(x_t) + p_t), \qquad \alpha_t = \frac{\exp(s_t)}{\sum_j \exp(s_j)},$$

then pools

$$h_{\text{var}}(x) = \sum_{t=1}^{T} \alpha_t (e(x_t) + p_t), \qquad p_{\text{var}}(y \mid x) = \operatorname{softmax}(W h_{\text{var}} + b).$$

For paired evaluation over `N` examples, define

$$\Delta_{\text{acc}} = \frac{1}{N} \sum_{i=1}^{N} \left[ \mathbf{1}(\hat{y}^{(i)}_{\text{var}} = y^{(i)}) - \mathbf{1}(\hat{y}^{(i)}_{\text{base}} = y^{(i)}) \right],$$

and

$$\Delta_{\text{nll}} = \frac{1}{N} \sum_{i=1}^{N} \left[ \ell_{\text{base}}^{(i)} - \ell_{\text{var}}^{(i)} \right], \qquad \ell^{(i)} = -\log p(y^{(i)} \mid x^{(i)}).$$

Positive `Delta_acc` and positive `Delta_nll` mean the variant improved on the same held-out examples.

## PyTorch implementation

The code below keeps the experiment bounded and deterministic. It uses the repo's quick profile only to set a small budget, not to build a full training pipeline.

In [ ]:
from pathlib import Path
import sys

import torch
from torch import nn
import torch.nn.functional as F

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.configs import profile_config, set_seed

set_seed(7)
quick_model_cfg, quick_train_cfg = profile_config('quick')
EXPERIMENT = {
    'vocab_size': 8,
    'seq_len': 6,
    'train_examples': 512,
    'eval_examples': 128,
    'steps': max(20, quick_train_cfg.max_steps // 5),
    'batch_size': max(64, quick_train_cfg.batch_size * 8),
    'embed_dim': min(16, quick_model_cfg.n_embd // 4),
}
EXPERIMENT

In [ ]:
def make_split(num_examples: int) -> tuple[torch.Tensor, torch.Tensor]:
    tokens = torch.randint(0, EXPERIMENT['vocab_size'], (num_examples, EXPERIMENT['seq_len']))
    labels = tokens[:, 0].clone()
    return tokens, labels

train_x, train_y = make_split(EXPERIMENT['train_examples'])
eval_x, eval_y = make_split(EXPERIMENT['eval_examples'])

class MeanPoolBaseline(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.embed = nn.Embedding(EXPERIMENT['vocab_size'], EXPERIMENT['embed_dim'])
        self.head = nn.Linear(EXPERIMENT['embed_dim'], EXPERIMENT['vocab_size'])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, None]:
        pooled = self.embed(x).mean(dim=1)
        return self.head(pooled), None

class AttentionPoolVariant(nn.Module):
    def __init__(self, use_positions: bool) -> None:
        super().__init__()
        self.embed = nn.Embedding(EXPERIMENT['vocab_size'], EXPERIMENT['embed_dim'])
        self.pos = nn.Embedding(EXPERIMENT['seq_len'], EXPERIMENT['embed_dim']) if use_positions else None
        self.query = nn.Parameter(torch.zeros(EXPERIMENT['embed_dim']))
        self.head = nn.Linear(EXPERIMENT['embed_dim'], EXPERIMENT['vocab_size'])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        hidden = self.embed(x)
        if self.pos is not None:
            positions = torch.arange(x.size(1), device=x.device)
            hidden = hidden + self.pos(positions)[None, :, :]
        scores = hidden @ self.query
        weights = torch.softmax(scores, dim=1)
        pooled = (weights.unsqueeze(-1) * hidden).sum(dim=1)
        return self.head(pooled), weights

def train_and_evaluate(model: nn.Module) -> dict[str, torch.Tensor | float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    for step in range(EXPERIMENT['steps']):
        start = (step * EXPERIMENT['batch_size']) % train_x.size(0)
        stop = start + EXPERIMENT['batch_size']
        batch_x = train_x[start:stop]
        batch_y = train_y[start:stop]
        optimizer.zero_grad()
        logits, _ = model(batch_x)
        loss = F.cross_entropy(logits, batch_y)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        logits, weights = model(eval_x)
        predictions = logits.argmax(dim=-1)
        correct = (predictions == eval_y).float()
        nll = F.cross_entropy(logits, eval_y, reduction='none')
    return {
        'accuracy': correct.mean().item(),
        'correct': correct,
        'nll': nll,
        'weights': weights,
    }

results = {
    'baseline': train_and_evaluate(MeanPoolBaseline()),
    'no_position_attention': train_and_evaluate(AttentionPoolVariant(use_positions=False)),
    'position_attention': train_and_evaluate(AttentionPoolVariant(use_positions=True)),
}

paired_accuracy_delta = (
    results['position_attention']['correct'] - results['baseline']['correct']
).mean().item()
paired_nll_gain = (
    results['baseline']['nll'] - results['position_attention']['nll']
).mean().item()

summary = {
    name: {
        'accuracy': round(payload['accuracy'], 4),
        'mean_nll': round(payload['nll'].mean().item(), 4),
    }
    for name, payload in results.items()
}
summary, {'paired_accuracy_delta': round(paired_accuracy_delta, 4), 'paired_nll_gain': round(paired_nll_gain, 4)}

## Numerical checks

A research notebook should check the mechanism, not just print a metric. Here we verify three things:

1. The baseline stays near the permutation-invariant ceiling for this task.
2. The positional attention variant improves on the same held-out examples.
3. The attention weights remain normalized and place most of their mass on the first position.

In [ ]:
variant_weights = results['position_attention']['weights']
assert variant_weights is not None
assert torch.allclose(variant_weights.sum(dim=1), torch.ones(eval_x.size(0)), atol=1e-6)
assert results['baseline']['accuracy'] < 0.6
assert results['position_attention']['accuracy'] > 0.95
assert paired_accuracy_delta > 0.25
assert paired_nll_gain > 0.5
assert variant_weights[:, 0].mean().item() > 0.8

{
    'baseline_accuracy': round(results['baseline']['accuracy'], 4),
    'no_position_accuracy': round(results['no_position_attention']['accuracy'], 4),
    'position_attention_accuracy': round(results['position_attention']['accuracy'], 4),
    'paired_accuracy_delta': round(paired_accuracy_delta, 4),
    'paired_nll_gain': round(paired_nll_gain, 4),
    'mean_attention_mass_on_position_0': round(variant_weights[:, 0].mean().item(), 4),
}

## Ablations

The tightest ablation is to keep attention pooling but remove positional embeddings. That makes the variant close to permutation-invariant again, so performance should collapse toward the baseline.

In this notebook the ablation already runs as `no_position_attention`. If the ablation matches the full variant, then the claimed mechanism was not the real reason for the gain.

In [ ]:
{
    'full_variant_accuracy': round(results['position_attention']['accuracy'], 4),
    'no_position_ablation_accuracy': round(results['no_position_attention']['accuracy'], 4),
    'accuracy_drop_without_positions': round(
        results['position_attention']['accuracy'] - results['no_position_attention']['accuracy'],
        4,
    ),
}

## Assumptions

- The synthetic task isolates order sensitivity rather than language modeling difficulty.
- The label is not leaked by token frequency or another shortcut.
- The optimization budget is enough for all compared models to fit their best simple solution.
- The paired evaluation split is identical across baseline, variant, and ablation.

## Risks

- A toy task can overstate how cleanly a mechanism transfers to real translation or language modeling.
- If the architecture change also changes parameter count too much, the comparison stops isolating the mechanism.
- Deterministic synthetic data can hide robustness issues that appear on noisy distributions.
- A single successful toy replication does not validate the original paper's full benchmark claim.

## Failure criteria

This notebook's main claim fails if any of the following happen:

- The positional attention variant does not beat the baseline by at least `+0.25` paired accuracy delta.
- The no-position ablation performs nearly as well as the full variant.
- The attention weights do not concentrate on the first position despite high claimed performance.
- Re-running with the same seed changes the qualitative conclusion.

## Exercises

Use the companion exercise files for written work:

- `exercises/research/13_research_orientation.md`
- `exercises/research/solutions/13_research_orientation_solutions.md`

Suggested prompts:

1. Replace the first-token label with the last-token label and predict what changes.
2. Swap the paired accuracy delta for a paired calibration or log-loss comparison.
3. Critique whether the toy replication actually supports the original paper claim or only a narrower mechanism claim.

## References

- Vaswani et al. *Attention Is All You Need* [@attention_is_all_you_need].